# Combined dataset inventory

* **Developed by:** Anna Maguza
* **Affilation:** CellZome, a GSK company
* **Created date:** 2026-05-08
* **Last modified date:** 2026-05-13

Cell counts and lgr5_status summary across atlas + LGR5 datasets.


## 1. Imports and paths

In [1]:
import os, sys, gc, json, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
sc.settings.verbosity = 2
sc.settings.dpi = 120
sc.settings.dpi_save = 300
plt.rcParams.update({
    'savefig.bbox': 'tight', 'savefig.dpi': 300, 'figure.dpi': 120,
    'font.family': ['Arial', 'Helvetica', 'DejaVu Sans'], 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

In [3]:
REPO         = Path('/Users/am336941/Library/CloudStorage/OneDrive-GSK/Desktop/Fetal_stem_cells_analysis')
DATA_OUT     = Path('/Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced')
ATLAS_PATH   = Path('/Users/am336941/PhD/data/gut_data/gut_hs_all_datasets_full_annotated_AM_30102025_181544_raw.h5ad')
LGR5_DIR     = REPO / 'data' / 'LGR5_analysis'
ORTH_PATH    = Path('/Users/am336941/PhD/data/LGR5_analysis_data/human_mouse_orthologues_ensembl.txt')
DATA_OUT.mkdir(parents=True, exist_ok=True)

In [4]:
def stamp() -> str:
    return dt.datetime.now().strftime('%d%m%Y_%H%M%S')

## 2. Per-dataset cell counts and library-prep summary

In [5]:
files = sorted(DATA_OUT.glob('gut_*_std_*_raw.h5ad'))
rows = []
for f in files:
    a = sc.read_h5ad(f)
    by_state = a.obs.get('cell_states', pd.Series(['unknown']*a.n_obs, index=a.obs.index)).value_counts().to_dict()
    rows.append({
        'file': f.name,
        'study_id': a.obs['study_id'].iloc[0] if 'study_id' in a.obs.columns else '?',
        'species': a.obs['species'].iloc[0] if 'species' in a.obs.columns else '?',
        'n_cells': a.n_obs,
        'n_genes': a.n_vars,
        'lgr5+': int((a.obs.get('lgr5_status','') == 'LGR5+').sum()) if 'lgr5_status' in a.obs.columns else 0,
        'lgr5-': int((a.obs.get('lgr5_status','') == 'LGR5-').sum()) if 'lgr5_status' in a.obs.columns else 0,
        'unknown': int((a.obs.get('lgr5_status','') == 'unknown').sum()) if 'lgr5_status' in a.obs.columns else 0,
        'cell_states': by_state,
    })
    del a
inv = pd.DataFrame(rows)
display(inv)


,file,study_id,species,n_cells,n_genes,lgr5+,lgr5-,unknown,cell_states
0,gut_hs_Ishikawa2022_std_13052026_144051_raw.h5ad,Ishikawa2022,human,7103,33538,108,0,6995,{'unknown': 7103}
1,gut_mm_GSE145866_std_13052026_144053_raw.h5ad,GSE145866,mouse,4815,27998,4815,0,0,{'unknown': 4815}
2,gut_mm_GSE62270_std_13052026_144055_raw.h5ad,GSE62270,mouse,288,23630,288,0,0,{'unknown': 288}
3,gut_mm_GSE62784_std_13052026_144055_raw.h5ad,GSE62784,mouse,16,18315,16,0,0,{'unknown': 16}
4,gut_mm_GSE92865_std_13052026_144055_raw.h5ad,GSE92865,mouse,13247,27998,11268,1979,0,{'unknown': 13247}
5,gut_mm_GSE99457_std_13052026_144058_raw.h5ad,GSE99457,mouse,5570,27998,1764,1756,2050,{'unknown': 5570}
6,gut_mm_Grün2016_std_13052026_144100_raw.h5ad,Grün2016,mouse,96,57186,96,0,0,{'unknown': 96}
7,gut_mm_Haber2017_std_13052026_144100_raw.h5ad,Haber2017,mouse,1522,20107,411,1111,0,{'unknown': 1522}


## 3. Atlas summary

In [6]:
atlas = sorted(DATA_OUT.glob('gut_hs_atlas_epithelial_AM_*_raw.h5ad'))[-1]
a = sc.read_h5ad(atlas)
display(a.obs.groupby(['age_group','cell_states','gut_region'], observed=True).size().unstack(fill_value=0))


gut_region                              large intestine  small intestine
age_group          cell_states                                          
adult              Stem cells                      1317             1230
                   TA                              1683              645
                   Enterocyte                      1710             2781
                   Proximal progenitor             3069              975
                   Colonocyte                      1506               36
cell culture model Stem cells                         0             3439
                   TA                                 0             4271
                   Enterocyte                         0             3031
                   Proximal progenitor                0             5029
                   Distal progenitor                  0               48
                   Colonocyte                         0              337
child stage        Stem cells                         0              201
                   TA                                 0              256
                   Enterocyte                         0              106
                   Proximal progenitor                0               57
                   Colonocyte                         0                3
first trimester    Stem cells                       409             3298
                   TA                               721             5882
                   Enterocyte                       154            11114
                   Proximal progenitor             1986            12455
                   Distal progenitor                174               14
                   Colonocyte                       104             3003
second trimester   Stem cells                       155             3446
                   TA                               115             1570
                   Enterocyte                       102             9753
                   Proximal progenitor              104             2629
                   Distal progenitor                 10                3
                   Colonocyte                       209             6217

## 4. Save inventory CSV

In [7]:
inv.to_csv(DATA_OUT / f'inventory_combined_{stamp()}.csv', index=False)
print('done')


done
